# Homework 4: Evaluation Metrics for Classification

* Dataset: https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

In [ ]:
data = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'

In [ ]:
!wget $data -O data-week-4.csv 

In [ ]:
df = pd.read_csv('data-week-4.csv')
df.head()

In [ ]:
df.info()

In [ ]:
# Check missing values.
df.isnull().sum()

In [ ]:
df.dtypes

In [ ]:
# Replace missing values from categorical features with 'NA'
categorical_cols = ['lead_source', 'industry', 'employment_status', 'location']
df[categorical_cols] = df[categorical_cols].fillna('NA')
df.isnull().sum()

In [ ]:
# Replace missing values from numerical features with 0.0
numerical_cols = ['annual_income', 'lead_score']
df[numerical_cols] = df[numerical_cols].fillna(0.0)
df.isnull().sum()

In [ ]:
# This safely targets both older 'object' text and newer 'string' text
categorical_columns = list(df.select_dtypes(include=['object', 'string']).columns)

for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_') # Standardizes object column data.

In [ ]:
df.head().T  # Transposes the first 5 rows of the DataFrame for easier vertical reading

In [ ]:
# Check unique values
for col in df.columns:
    print(col)
    print(df[col].unique()[:5])
    print(df[col].nunique())
    print()

In [ ]:
numerical_columns = ['annual_income', 'interaction_count', 'number_of_courses_viewed', 'lead_score']

In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold

In [ ]:
#Evaluation Homework

In [ ]:
#question -1
auc_scores = {}

for feature in numerical_columns:
    # Calculate initial ROC AUC using the raw feature values
    auc = roc_auc_score(y_train, df_train[feature])

    # If AUC < 0.5, the feature is negatively correlated with the target.
    # Invert it (-) to flip the direction, making the score meaningful for ranking.
    # Negative correlation still has predictive power
    if auc < 0.5:
        auc = roc_auc_score(y_train, -df_train[feature])

    auc_scores[feature] = auc

auc_scores

In [ ]:
#question-2
y_pred = model.predict_proba(X_val)[:, 1]

auc = roc_auc_score(y_val, y_pred)

round(auc, 3)

In [ ]:
#question-3
thresholds = np.linspace(0, 1, 101)

scores = []

for threshold in thresholds:
    actual_positive = y_val == 1
    actual_negative = y_val == 0

    predict_positive = y_pred >= threshold
    predict_negative = y_pred < threshold

    tp = (predict_positive & actual_positive).sum()
    tn = (predict_negative & actual_negative).sum()
    fp = (predict_positive & actual_negative).sum()
    fn = (predict_negative & actual_positive).sum()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0 # Precision : From the `pre`dicted positives, how many we predicted right out of our positive predictions
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0 # Recall : From the `re` `al` positives, how many we predicted right out of the actual true occurences 

    scores.append((threshold, precision, recall))

columns = ["threshold", "precision", "recall"]

df_scores = pd.DataFrame(scores, columns=columns)

df_scores.head()

In [ ]:
#question-4
plt.figure(figsize=(10, 5))

# Plot precision and recall
plt.plot(
    df_scores["threshold"],
    df_scores["precision"],
    label="Precision",
    linewidth=2
)
plt.plot(
    df_scores["threshold"],
    df_scores["recall"],
    label="Recall",
    linewidth=2
)

intersect_threshold = 0.63

plt.axvline(
    x=intersect_threshold, 
    color='gray', 
    linestyle='--', 
    alpha=0.7, 
    label=f'Intersect (~{intersect_threshold})'
)

# Add more granular markings/ticks on the axes
plt.xticks(np.arange(0, 1.1, 0.1))  # Marks every 0.1 from 0 to 1 on the X-axis
plt.yticks(np.arange(0, 1.1, 0.1))  # Marks every 0.1 from 0 to 1 on the Y-axis

# Labels, title, grid, and legend
plt.xlabel("Threshold", fontsize=12)
plt.ylabel("Score", fontsize=12)
plt.title("Precision and Recall at Different Thresholds", fontsize=14)

plt.grid(True, linestyle=":", alpha=0.6)
plt.ylim(0, 1.05)
plt.legend(fontsize=11)

plt.show()

In [ ]:
# 1. Filter out dead zones: remove rows where BOTH precision and recall are 0.
# The tilde (~) means "NOT", so we keep all other valid thresholds.
df_nonzero = df_scores[
    ~((df_scores["precision"] == 0) & (df_scores["recall"] == 0))
].copy()

# 2. Calculate the absolute gap between precision and recall for each threshold.
df_nonzero["difference"] = (
    df_nonzero["precision"] - df_nonzero["recall"]
).abs()

# 3. Find the threshold where precision and recall are closest (smallest difference).
# .idxmin() gets the index of the minimum difference, and .loc retrieves that row.
df_nonzero.loc[
    df_nonzero["difference"].idxmin()
]

In [ ]:
#question -4
df_scores["f1"] = (
    2
    * df_scores["precision"]
    * df_scores["recall"]
    / (df_scores["precision"] + df_scores["recall"])
).fillna(0)

df_scores.head()

In [ ]:
df_scores.loc[df_scores["f1"].idxmax()]

In [ ]:
#quest-5
y_full_train = df_full_train['converted'].values
df_full_train_X = df_full_train.drop(columns=['converted'])

In [ ]:
kfold = KFold(n_splits=5, shuffle=True, random_state=1)

auc_scores_cv = []

for train_idx, val_idx in kfold.split(df_full_train_X):
    df_train_fold = df_full_train_X.iloc[train_idx]
    df_val_fold = df_full_train_X.iloc[val_idx]

    y_train_fold = y_full_train[train_idx]
    y_val_fold = y_full_train[val_idx]

    dv_fold = DictVectorizer(sparse=False)
    X_train_fold = dv_fold.fit_transform(df_train_fold.to_dict(orient="records"))
    X_val_fold = dv_fold.transform(df_val_fold.to_dict(orient="records"))

    model_fold = LogisticRegression(solver="liblinear", C=1.0, max_iter=1000, random_state=1)
    model_fold.fit(X_train_fold, y_train_fold)

    y_pred_fold = model_fold.predict_proba(X_val_fold)[:, 1]
    auc_scores_cv.append(roc_auc_score(y_val_fold, y_pred_fold))

auc_scores_cv

In [ ]:
round(np.std(auc_scores_cv), 3)

In [ ]:
#question-6
kfold = KFold(n_splits=5, shuffle=True, random_state=1)

for C in [0.000001, 0.001, 1]:
    auc_scores_c = []

    for train_idx, val_idx in kfold.split(df_full_train_X):
        df_train_fold = df_full_train_X.iloc[train_idx]
        df_val_fold = df_full_train_X.iloc[val_idx]

        y_train_fold = y_full_train[train_idx]
        y_val_fold = y_full_train[val_idx]

        dv_fold = DictVectorizer(sparse=False)
        X_train_fold = dv_fold.fit_transform(df_train_fold.to_dict(orient="records"))
        X_val_fold = dv_fold.transform(df_val_fold.to_dict(orient="records"))

        model_fold = LogisticRegression(solver="liblinear", C=C, max_iter=1000, random_state=1)
        model_fold.fit(X_train_fold, y_train_fold)

        y_pred_fold = model_fold.predict_proba(X_val_fold)[:, 1]
        auc_scores_c.append(roc_auc_score(y_val_fold, y_pred_fold))

    print(f"C={C:<8} Mean AUC: {np.mean(auc_scores_c):.3f} ± {np.std(auc_scores_c):.3f}")

KeyboardInterrupt: 